In [ ]:
# import os
# from huggingface_hub import snapshot_download

# # make sure offline mode is off for this one download
# os.environ.pop("HF_HUB_OFFLINE", None)

# path = snapshot_download("Qwen/Qwen2.5-7B-Instruct", cache_dir="/scratch/gandhi.gau/hf_cache/hub")

# print("Downloaded to:", path)
# print("Version (snapshot):", os.path.basename(path))   

/home/gandhi.gau/.conda/envs/my_gpu_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 14 files: 100%|██████████| 14/14 [01:25<00:00,  6.12s/it]

Downloaded to: /scratch/gandhi.gau/hf_cache/hub/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28
Version (snapshot): a09a35458c702b33eeacc393d103063234e8bc28


In [ ]:
# import glob, os

# snap = "/scratch/gandhi.gau/hf_cache/hub/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28"

# files = sorted(os.listdir(snap))
# print("files:", files)

# shards = glob.glob(f"{snap}/*.safetensors")
# total_gb = sum(os.path.getsize(f) for f in shards) / 1e9
# print("weight shards:", len(shards), "| total size:", round(total_gb, 1), "GB")

files: ['.gitattributes', 'LICENSE', 'README.md', 'config.json', 'generation_config.json', 'merges.txt', 'model-00001-of-00004.safetensors', 'model-00002-of-00004.safetensors', 'model-00003-of-00004.safetensors', 'model-00004-of-00004.safetensors', 'model.safetensors.index.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']
weight shards: 4 | total size: 15.2 GB


In [ ]:
# one-time install of the two metric libraries (skipped if they are already there)
import importlib.util, subprocess, sys
for pkg, name in [("bert-score", "bert_score"), ("sacrebleu", "sacrebleu"), ("nltk", "nltk")]:
    if importlib.util.find_spec(name) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
        

In [ ]:
import os
os.environ["USE_TF"] = "0"                            # stops TensorFlow warning messages
os.environ["HF_HOME"] = "/scratch/gandhi.gau/hf_cache"     # shared cache  (read-only for us)
os.environ["HF_HUB_OFFLINE"] = "1"                         # only load from the cache, never download or write to it
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import re, json, glob, time, pickle, datetime
import numpy as np
import pandas as pd
import torch
import transformers
from scipy.stats import binomtest
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve, confusion_matrix
import matplotlib.pyplot as plt

# record which GPU this run used
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
DATA_USER = "gandhi.gau"           # inputs (model cache, data splits, DeBERTa models) are read from Vaish's folders
RUN_USER = os.environ["USER"]      # outputs are saved in the folders of whoever runs this

CACHE = f"/scratch/{DATA_USER}/hf_cache/hub"
SPLITS = f"/home/{DATA_USER}/NLP_Project/paraphrase/splits"
DEBERTA_OUT = f"/scratch/{DATA_USER}/paraphrase/out_deberta"      # saved DeBERTa models
OUT = f"/home/{RUN_USER}/NLP_Project/qwen_generation/taskD"
os.makedirs(OUT, exist_ok=True)

# the exact Qwen version we downloaded
HF_ID = "Qwen/Qwen2.5-7B-Instruct"
REVISION = "a09a35458c702b33eeacc393d103063234e8bc28"

SEED = 42
SEEDS = [42, 43, 44, 45, 46]     # the 5 DeBERTa runs

In [ ]:
t0 = time.time()
gtok = AutoTokenizer.from_pretrained(HF_ID, revision=REVISION, cache_dir=CACHE,
                                     local_files_only=True, padding_side="left")
gen_model = AutoModelForCausalLM.from_pretrained(HF_ID, revision=REVISION, cache_dir=CACHE,
                                                 local_files_only=True, torch_dtype=torch.bfloat16,
                                                 device_map="cuda").eval()
print("Qwen loaded in", round(time.time() - t0), "s |", round(torch.cuda.memory_allocated() / 1e9, 1), "GB")

In [ ]:
# Qwen sometimes switches to Chinese, so we block every Chinese/Japanese/Korean token.
# The list is saved the first time, so later runs just load it.
SUPPRESS_FILE = f"{OUT}/qwen_cjk_suppress.pkl"     # kept in our own output folder, since the cache is read-only

if os.path.exists(SUPPRESS_FILE):
    SUPPRESS = pickle.load(open(SUPPRESS_FILE, "rb"))
else:
    CJK = re.compile(r"[\u3040-\u30ff\u3400-\u9fff\uac00-\ud7af\uff00-\uffef]")
    special = set(gtok.all_special_ids)
    texts = gtok.batch_decode([[i] for i in range(len(gtok))])     # all tokens in one call
    SUPPRESS = [i for i, t in enumerate(texts) if i not in special and CJK.search(t)]
    pickle.dump(SUPPRESS, open(SUPPRESS_FILE, "wb"))

print("blocked tokens:", len(SUPPRESS))

In [ ]:
# same sampling settings as the rest of the team
GEN_CFG = dict(temperature=0.7, top_p=0.9, top_k=40, max_new_tokens=128)

def generate_batch(system_prompt, inputs, temperature=None):
    """
    Send a batch of sentences to Qwen with the same system prompt and return its replies.
    """
    prompts = []
    for text in inputs:
        msgs = [{"role": "system", "content": system_prompt}, {"role": "user", "content": text}]
        prompts.append(gtok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))

    # same settings as the team, unless a different temperature is asked for
    cfg = dict(GEN_CFG)
    if temperature is not None:
        cfg["temperature"] = temperature

    enc = gtok(prompts, return_tensors="pt", padding=True).to("cuda")
    with torch.no_grad():
        out = gen_model.generate(**enc, do_sample=True, pad_token_id=gtok.pad_token_id,
                                 suppress_tokens=SUPPRESS, **cfg)

    # keep only the new tokens, not the prompt
    new = out[:, enc.input_ids.shape[1]:]
    return [t.strip() for t in gtok.batch_decode(new, skip_special_tokens=True)]

torch.manual_seed(SEED)
print(generate_batch("Paraphrase the sentence. Output only the paraphrase.", ["The cat sat on the mat."])[0])

In [ ]:
snap = f"{CACHE}/models--Qwen--Qwen2.5-7B-Instruct/snapshots/{REVISION}"
cfg = json.load(open(f"{snap}/config.json"))

print("model:        ", HF_ID)
print("revision:     ", REVISION)
print("transformers: ", transformers.__version__, "| torch:", torch.__version__)
print("parameters:   ", f"{sum(p.numel() for p in gen_model.parameters()) / 1e9:.2f}B")
for k in ["num_hidden_layers", "hidden_size", "num_attention_heads", "num_key_value_heads",
          "vocab_size", "max_position_embeddings", "torch_dtype"]:
    print(f"{k:24s}", cfg.get(k))

# Task D: Near-miss sentences (hard negatives)

In this notebook, Qwen takes a real sentence from our test sets and makes a small edit that changes its meaning while keeping almost all of its words. For example, "Anna emailed Ben" becomes "Ben emailed Anna". This is the same kind of trap as the PAWS non-paraphrases, so every pair we create here has the true label 0 (not a paraphrase).

We then check two things:
1. Did Qwen do the job: Filter out pairs where the edit didn't happen, and measure how similar the new sentence is to the original (BERTScore and BLEU).
2. Do the classifiers get fooled: Score every pair with DeBERTa (our winning model), the debiased DeBERTa and Qwen, and count how often each one wrongly calls the pair a paraphrase. We call this the fool rate. A lower fool rate is better.

Everything is zero-shot, so no model is trained here. We only use the test splits.

## Choosing the source sentences
We start from `sentence1` of the label-1 test pairs, because these are normal, well-formed sentences that have a real paraphrase in the dataset. We use all 279 from MRPC and a random 1,000 each from QQP and PAWS, so no single dataset dominates the results.

We keep each sentence's row number (`idx`) in its test file, so these pairs can later be matched with the Task A paraphrases of the same sentences.

In [ ]:
rows = []
for ds, n in [("mrpc", None), ("qqp", 1000), ("paws", 1000)]:
    df = pd.read_csv(f"{SPLITS}/{ds}_test.csv", keep_default_na=False)
    df = df.reset_index().rename(columns={"index": "idx"})    # row number in the test file

    pos = df[df["label"] == 1]
    if n is not None and len(pos) > n:
        pos = pos.sample(n, random_state=SEED)                 # same sample every time

    for _, r in pos.iterrows():
        rows.append({"dataset": ds, "idx": r["idx"], "source": str(r["sentence1"])})

src = pd.DataFrame(rows)
print(src.groupby("dataset").size())

## Three kinds of edits
To make the near-misses varied (and closer to PAWS), we ask Qwen for one of three edits:
- **number:** change one number, date or amount (e.g. "born in 1970" -> "born in 1971")
- **swap:** swap two names, places, objects or roles (e.g. "Anna emailed Ben" -> "Ben emailed Anna")
- **negation:** flip the meaning by adding or removing a "not" or "never"

Sentences that contain a number get the number edit. The rest alternate between swap and negation. Each prompt tells Qwen to change as few words as possible, and to reply "NONE" if the edit doesn't make sense for that sentence. We remove those later.

In [ ]:
COMMON = (" Keep every other word the same and keep the same style. "
          "A statement must stay a statement, and a question must stay a question. "
          "The result must be grammatical and natural. "
          "If this change is not possible, output only NONE. "
          "Write in English and output only the new sentence.")

PROMPTS = {
    "number": "Change one number, date or amount in the sentence so that its meaning changes." + COMMON,
    "swap": ("Swap the positions of two names, places, objects or roles that are already in the sentence, "
             "so that its meaning changes. Do not replace them with new words." + COMMON),
    "negation": "Add or remove one negation, such as 'not' or 'never', so that the sentence means the opposite." + COMMON,
}

# decide which edit each sentence gets
def pick_edit(position, sentence):
    if re.search(r"\d", sentence):
        return "number"
    return "swap" if position % 2 == 0 else "negation"

src["edit_type"] = [pick_edit(i, s) for i, s in enumerate(src["source"])]
print(pd.crosstab(src["dataset"], src["edit_type"]))

In [ ]:
# small test before the full run: 2 sentences per edit type
test = src.groupby("edit_type").head(2)
for edit, group in test.groupby("edit_type"):
    torch.manual_seed(SEED)
    outs = generate_batch(PROMPTS[edit], group["source"].tolist())
    for s, o in zip(group["source"], outs):
        print(f"[{edit}]\n  SRC: {s}\n  NEW: {o}\n")

## Generating the near-misses
We send the sentences in batches of 128, grouped by edit type so each batch uses one prompt. Each batch has its own fixed seed, so the run can be repeated exactly. If the output file already exists, the cell loads it instead of generating again.

In [ ]:
GEN_FILE = f"{OUT}/taskD_generated.csv"
BS = 128

if os.path.exists(GEN_FILE):
    gen = pd.read_csv(GEN_FILE, keep_default_na=False)
    print("Loaded saved generations:", len(gen))
else:
    parts = []
    batch_no = 0
    t0 = time.time()
    for edit, group in src.groupby("edit_type"):
        group = group.copy()
        outputs = []
        for i in range(0, len(group), BS):
            torch.manual_seed(SEED + batch_no)          # fixed seed for each batch
            outputs += generate_batch(PROMPTS[edit], group["source"].iloc[i:i + BS].tolist())
            batch_no += 1
        group["near_miss"] = outputs
        parts.append(group)
        print(f"{edit}: {len(group)} done")

    gen = pd.concat(parts).sort_values(["dataset", "idx"]).reset_index(drop=True)
    gen.to_csv(GEN_FILE, index=False)
    print(f"Generated {len(gen)} sentences in {time.time() - t0:.0f} s")


In [ ]:
# save how this was generated
meta = {"task": "D: near-miss hard negatives", "model": HF_ID, "revision": REVISION,
        "dtype": "bfloat16", "gen_config": GEN_CFG, "seed": SEED, "batch_size": BS,
        "prompts": PROMPTS, "date": datetime.datetime.now().isoformat(timespec="seconds")}
json.dump(meta, open(f"{OUT}/taskD_metadata.json", "w"), indent=1)

## Cleaning and filtering
Not every output is a usable near-miss, so we remove a pair if:
- the output is empty, "NONE", an exact copy of the source, or starts with extra text like "Here is…"
- the sentence type changed (a question became a statement, or a statement became a question)
- the length changed a lot (less than 0.7 or more than 1.4 times the source length)
- the two sentences share less than 70% of their words (Jaccard overlap), because then it is no longer a near-miss
- the requested edit didn't actually happen:
  - **number:** none of the numbers changed (we ignore commas, so "1,000" and "1000" count as the same)
  - **negation:** the meaning didn't flip. We count the negation words (not, never, no, n't, …), and the count has to go up or down by an odd number. So "not" -> "never" doesn't count as a flip.
  - **swap:** the words changed too much, or nothing actually moved (the words are still in the same order)

Before filtering, we also tidy the formatting. MRPC and PAWS are written in Penn Treebank style, with spaces around punctuation and split contractions, but Qwen writes normal text. A classifier could pick up on that difference instead of the meaning, so we run Qwen's sentences through NLTK's standard Treebank tokenizer to match the dataset style. QQP is already normal text, so its sentences stay as they are.

In [ ]:
from nltk.tokenize import TreebankWordTokenizer
treebank = TreebankWordTokenizer()

def jaccard(s1, s2):
    # shared words / all unique words, same as in our EDA and baseline
    w1 = set(re.findall(r"\w+", s1.lower()))
    w2 = set(re.findall(r"\w+", s2.lower()))
    if len(w1 | w2) == 0:
        return 0.0
    return len(w1 & w2) / len(w1 | w2)

def normalise(text, ds):
    # MRPC and PAWS are written in Penn Treebank style,
    # so we put Qwen's sentence through the standard Treebank tokenizer to match it
    if ds in ("mrpc", "paws"):
        text = " ".join(treebank.tokenize(text))
        if ds == "mrpc":
            # MRPC keeps plain double quotes instead of `` and ''
            text = text.replace("``", '"').replace("''", '"')
    return " ".join(text.split())

In [ ]:
NEG_WORDS = {"not", "no", "never", "none", "nobody", "nothing", "neither", "nor", "cannot", "without"}

def negations(s):
    # negation words, plus words ending in n't
    words = re.findall(r"[\w']+", s.lower())
    return [w for w in words if w in NEG_WORDS or w.endswith("n't")]

In [ ]:
# make sure every output is text, then match MRPC/PAWS spacing style
gen["near_miss"] = gen["near_miss"].astype(str)
gen["near_miss_clean"] = [normalise(t, ds) for t, ds in zip(gen["near_miss"], gen["dataset"])]

# s = original sentence, g = Qwen's edited sentence (extra spaces at the ends removed)
s = gen["source"].str.strip()
g = gen["near_miss_clean"].str.strip()

# one True or False column per check (True = problem found)
flags = pd.DataFrame(index=gen.index)

# Qwen returned nothing, or said NONE because the edit wasn't possible
flags["empty_or_none"] = (g == "") | (g.str.upper().str.strip(". ") == "NONE")

# Qwen returned the same sentence, so nothing changed
flags["copy"] = g.str.lower() == s.str.lower()

# Qwen added chatty text like "Sure, here is..." or "New sentence:"
flags["extra_text"] = g.str.contains(r"^(?:here|sure|certainly)\b|paraphrase:|sentence:", case=False, regex=True)

# the sentence type changed (a question became a statement, or a statement became a question)
flags["type_changed"] = s.str.endswith("?") != g.str.endswith("?")

# output is much shorter or longer than the source 
ratio = g.str.len() / s.str.len()
flags["length_change"] = (ratio < 0.7) | (ratio > 1.4)

# word overlap with the source, below 0.7 it's no longer a near-miss
gen["overlap"] = [jaccard(a, b) for a, b in zip(s, g)]
flags["low_overlap"] = gen["overlap"] < 0.7

## Checking if the edit happened

In [ ]:
def numbers(s):
    # numbers in the sentence, with commas removed so "1,000" and "1000" count as the same
    return re.findall(r"\d+(?:\.\d+)?", s.replace(",", ""))

def words(s):
    return re.findall(r"\w+", s.lower())

In [ ]:
# did the requested edit actually happen?
edit_missing = []
for edit, a, b in zip(gen["edit_type"], s, g):
    if edit == "number":
        # at least one number must be different
        edit_missing.append(numbers(a) == numbers(b))
    elif edit == "negation":
        # meaning only flips if the number of negations goes up or down by an odd amount
        # ("not" -> "never" or "cannot" -> "can not" do not count)
        edit_missing.append((len(negations(a)) - len(negations(b))) % 2 == 0)
    else:
        # swap: almost the same words, but in a different order
        edit_missing.append(jaccard(a, b) < 0.85 or words(a) == words(b))
flags["edit_missing"] = edit_missing

In [ ]:
# drop old flag columns first, so re-running this cell doesn't duplicate them
gen = gen.drop(columns=list(flags.columns) + ["keep"], errors="ignore")
gen["keep"] = ~flags.any(axis=1)
gen = pd.concat([gen, flags], axis=1)
gen.to_csv(f"{OUT}/taskD_all_with_flags.csv", index=False)

In [ ]:
# how many pairs each check removed, per dataset
summary = flags.groupby(gen["dataset"]).sum()
summary["kept"] = gen.groupby("dataset")["keep"].sum()
summary["total"] = gen.groupby("dataset").size()
summary["kept_%"] = (summary["kept"] / summary["total"] * 100).round(1)
summary

In [ ]:
clean = gen[gen["keep"]].reset_index(drop=True)
clean["label"] = 0      # every near-miss is a non-paraphrase by design
clean.to_csv(f"{OUT}/taskD_clean.csv", index=False)

print("Usable near-miss pairs:", len(clean))
print(pd.crosstab(clean["dataset"], clean["edit_type"], margins=True))

# a few random examples 
for _, r in clean.sample(6, random_state=SEED).iterrows():
    print(f"\n[{r.dataset} | {r.edit_type}]\n  SRC: {r.source}\n  NEW: {r.near_miss_clean}")

## check
The filters only catch obvious failures. To see how often the meaning really changed, we check 60 pairs by hand (20 per dataset). The cell below saves them to `taskD_handcheck.csv` with an empty `meaning_changed` column. We fill it with 1 (meaning changed) or 0 (still means the same), then rerun the next cell to get the result. The file is never overwritten, so the answers are safe.

In [ ]:
HAND_FILE = f"{OUT}/taskD_handcheck.csv"

if not os.path.exists(HAND_FILE):
    # 20 random pairs from each dataset
    sample = clean.groupby("dataset").sample(n=20, random_state=SEED)
    sample = sample[["dataset", "idx", "edit_type", "source", "near_miss_clean"]].copy()
    sample["meaning_changed"] = ""
    sample.to_csv(HAND_FILE, index=False)
    print("Saved", len(sample), "pairs to", HAND_FILE, "for checking")
    
hand = pd.read_csv(HAND_FILE, keep_default_na=False)
filled = hand[hand["meaning_changed"].astype(str).str.strip() != ""]
if len(filled) > 0:
    filled = filled.assign(meaning_changed=filled["meaning_changed"].astype(int))
    print(f"Checked {len(filled)} pairs: meaning really changed in {filled['meaning_changed'].mean():.1%}")
    display(filled.groupby(["dataset", "edit_type"])["meaning_changed"].mean().round(2))
else:
    print("Hand check not filled in yet")

## How close is each near-miss to its source?
We use the two automatic metrics agreed for Task D:
- **BLEU**: (0 to 100) counts shared word sequences. A good near-miss should score high, because it keeps most of the words.
- **BERTScore F1**: compares meaning with a pretrained encoder (RoBERTa-large). Since the meaning changed, we would like it to be lower than for a real paraphrase. It may still be high, though, because BERTScore is known to be less sensitive to small edits like swaps.

In [ ]:
import sacrebleu
from bert_score import score as bert_score

# BLEU of each near-miss against its source
clean["bleu"] = [sacrebleu.sentence_bleu(b, [a]).score for a, b in zip(clean["source"], clean["near_miss_clean"])]

# BERTScore F1 of each near-miss against its source
_, _, F1 = bert_score(clean["near_miss_clean"].tolist(), clean["source"].tolist(),
                      lang="en", batch_size=64, device="cuda", verbose=False)
clean["bertscore_f1"] = F1.numpy()

In [ ]:
metrics_table = clean.groupby(["dataset", "edit_type"])[["overlap", "bleu", "bertscore_f1"]].mean().round(3)
metrics_table.to_csv(f"{OUT}/taskD_text_metrics.csv")
metrics_table

## If the classifiers get fooled or not
Every pair here is a non-paraphrase, so any "paraphrase" prediction is a mistake. We score the pairs with:
- **DeBERTa (in-domain):** our winning model, using the version trained on the same dataset the sentence came from (5 seeds)
- **DeBERTa (QQP)** and **debiased DeBERTa (QQP):** both trained on QQP, so this is a direct test of whether debiasing also helps on brand-new trap pairs (5 seeds each)
- **Qwen (zero-shot):** the probability of answering "Yes" when asked whether the two sentences mean the same

A pair counts as "fooled" when the predicted probability of paraphrase is above 0.5, the same cut-off we used everywhere else.

One caveat: Qwen also wrote these sentences, so it is judging its own output, which may make it look better than it really is.

In [ ]:
# find the saved DeBERTa models (standard runs for each dataset, plus the debiased QQP runs)
def find_checkpoint(ds, seed, debiased=False):
    if debiased:
        path = f"{DEBERTA_OUT}/deberta_qqp_s{seed}_debias/best"
        return path if os.path.isdir(path) else None
    found = [p for p in glob.glob(f"{DEBERTA_OUT}/deberta_{ds}_s{seed}*/best")
             if "debias" not in p and "poe" not in p]
    return sorted(found)[0] if found else None

checkpoints = {}
for ds in ["mrpc", "qqp", "paws"]:
    for s in SEEDS:
        checkpoints[(ds, s)] = find_checkpoint(ds, s)
for s in SEEDS:
    checkpoints[("qqp_debias", s)] = find_checkpoint("qqp", s, debiased=True)

for key, path in checkpoints.items():
    print(key, "->", path)
assert all(checkpoints.values()), "Some DeBERTa checkpoints are missing, see the list above"

In [ ]:
# DeBERTa tokenizer, same settings as in training (pairs cut at 128 tokens)
deb_tok = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base", cache_dir=CACHE, local_files_only=True)

def deberta_probs(ckpt, s1, s2, bs=64):
    """
    P(paraphrase) from one saved DeBERTa model for a list of sentence pairs.
    """
    model = AutoModelForSequenceClassification.from_pretrained(ckpt).float().cuda().eval()
    probs = []
    for i in range(0, len(s1), bs):
        enc = deb_tok(s1[i:i + bs], s2[i:i + bs], truncation=True, max_length=128,
                      padding=True, return_tensors="pt").to("cuda")
        with torch.no_grad():
            logits = model(**enc).logits.float()
        probs += torch.softmax(logits, dim=-1)[:, 1].cpu().tolist()

    # free GPU memory before loading the next model
    del model
    torch.cuda.empty_cache()
    return np.array(probs)

In [ ]:
s1 = clean["source"].tolist()
s2 = clean["near_miss_clean"].tolist()

# 1) in-domain DeBERTa: each sentence is scored by the model trained on its own dataset
for s in SEEDS:
    col = np.zeros(len(clean))
    for ds in ["mrpc", "qqp", "paws"]:
        mask = (clean["dataset"] == ds).values
        idx = np.where(mask)[0]
        col[idx] = deberta_probs(checkpoints[(ds, s)], [s1[i] for i in idx], [s2[i] for i in idx])
    clean[f"deb_in_s{s}"] = col

In [ ]:
# 2) QQP-trained standard and debiased DeBERTa on every pair
for s in SEEDS:
    clean[f"deb_qqp_s{s}"] = deberta_probs(checkpoints[("qqp", s)], s1, s2)
    clean[f"deb_debias_s{s}"] = deberta_probs(checkpoints[("qqp_debias", s)], s1, s2)
    print("seed", s, "done")

In [ ]:
# average probability over the 5 seeds for each DeBERTa version
for name in ["deb_in", "deb_qqp", "deb_debias"]:
    clean[f"{name}_prob"] = clean[[f"{name}_s{s}" for s in SEEDS]].mean(axis=1)

For Qwen as a judge, we read the probability of "Yes" straight from the model's output scores instead of generating text. This gives a probability like DeBERTa's, so both can use the same 0.5 cut-off.

**Note:** for a fair comparison with our earlier Qwen results, `JUDGE_PROMPT` should be the exact prompt used in the team's Qwen classification run. Replace it below if it differs.

In [ ]:
JUDGE_PROMPT = ("Do the following two sentences have the same meaning? "
                "Answer with only Yes or No.\n\nSentence 1: {s1}\nSentence 2: {s2}")

YES_ID = gtok.encode("Yes", add_special_tokens=False)[0]
NO_ID = gtok.encode("No", add_special_tokens=False)[0]

def qwen_probs(s1, s2, bs=32):
    """
    P(Yes) from Qwen for each pair, using only the Yes and No scores of its first answer token.
    """
    probs = []
    for i in range(0, len(s1), bs):
        prompts = []
        for a, b in zip(s1[i:i + bs], s2[i:i + bs]):
            msgs = [{"role": "user", "content": JUDGE_PROMPT.format(s1=a, s2=b)}]
            prompts.append(gtok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))

        enc = gtok(prompts, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            last = gen_model(**enc).logits[:, -1, :]          # scores for the next token (left padding, so -1 is correct)
        two = last[:, [YES_ID, NO_ID]].float()
        probs += torch.softmax(two, dim=-1)[:, 0].cpu().tolist()
    return np.array(probs)

clean["qwen_prob"] = qwen_probs(s1, s2)
clean.to_csv(f"{OUT}/taskD_scores.csv", index=False)
print("Scores saved for", len(clean), "pairs")

### Control: real paraphrases of the same sentences
A low fool rate only means something if the model still accepts real paraphrases. Otherwise a model that always says "no" would look perfect. Every source sentence we used also has a human-written paraphrase in its dataset (`sentence2`, label 1), so we score those pairs with the same judges. A good model should accept the real paraphrases and reject the near-misses.

In [ ]:
# the dataset's own human paraphrase (sentence2) for each source sentence
ref = []
for ds in ["mrpc", "qqp", "paws"]:
    df = pd.read_csv(f"{SPLITS}/{ds}_test.csv", keep_default_na=False)
    df = df.reset_index().rename(columns={"index": "idx"})
    ref.append(df[["idx", "sentence2"]].assign(dataset=ds))
clean = clean.merge(pd.concat(ref), on=["dataset", "idx"], how="left")
r2 = clean["sentence2"].tolist()

## We compute BLEU and BERTScore for the real paraphrases too, to see whether the metrics can tell a real paraphrase from a near-miss

In [ ]:
# same two metrics for the real human paraphrases, as a comparison
clean["ref_bleu"] = [sacrebleu.sentence_bleu(b, [a]).score for a, b in zip(clean["source"], clean["sentence2"])]

_, _, F1r = bert_score(clean["sentence2"].tolist(), clean["source"].tolist(),
                       lang="en", batch_size=64, device="cuda", verbose=False)
clean["ref_bertscore_f1"] = F1r.numpy()

# near-miss vs real paraphrase, per dataset
compare = clean.groupby("dataset")[["bleu", "ref_bleu", "bertscore_f1", "ref_bertscore_f1"]].mean().round(3)
compare.loc["all"] = clean[["bleu", "ref_bleu", "bertscore_f1", "ref_bertscore_f1"]].mean().round(3)
compare.to_csv(f"{OUT}/taskD_text_metrics_vs_real.csv")
compare

In [ ]:
# score the real paraphrase pairs with the same judges
for s in SEEDS:
    col = np.zeros(len(clean))
    for ds in ["mrpc", "qqp", "paws"]:
        idx = np.where((clean["dataset"] == ds).values)[0]
        col[idx] = deberta_probs(checkpoints[(ds, s)], [s1[i] for i in idx], [r2[i] for i in idx])
    clean[f"ref_deb_in_s{s}"] = col
    clean[f"ref_deb_qqp_s{s}"] = deberta_probs(checkpoints[("qqp", s)], s1, r2)
    clean[f"ref_deb_debias_s{s}"] = deberta_probs(checkpoints[("qqp_debias", s)], s1, r2)
clean["ref_qwen_prob"] = qwen_probs(s1, r2)
clean.to_csv(f"{OUT}/taskD_scores.csv", index=False)

In [ ]:
# side by side: accepts real paraphrases (higher is better) vs fooled by near-misses (lower is better)
rows = []
for name, label in [("deb_in", "DeBERTa (in-domain)"), ("deb_qqp", "DeBERTa (QQP)"),
                    ("deb_debias", "DeBERTa debiased (QQP)")]:
    accept = [(clean[f"ref_{name}_s{s}"] > 0.5).mean() for s in SEEDS]
    fooled = [(clean[f"{name}_s{s}"] > 0.5).mean() for s in SEEDS]
    rows.append({"judge": label,
                 "accepts real paraphrases": f"{np.mean(accept):.3f} ± {np.std(accept, ddof=1):.3f}",
                 "fooled by near-misses": f"{np.mean(fooled):.3f} ± {np.std(fooled, ddof=1):.3f}"})
rows.append({"judge": "Qwen (zero-shot)",
             "accepts real paraphrases": f"{(clean['ref_qwen_prob'] > 0.5).mean():.3f}",
             "fooled by near-misses": f"{(clean['qwen_prob'] > 0.5).mean():.3f}"})

In [ ]:
control = pd.DataFrame(rows)
control.to_csv(f"{OUT}/taskD_control.csv", index=False)
control

## Results: fool rate
The fool rate is the share of near-miss pairs a model calls a paraphrase. For DeBERTa we report the mean ± std over the 5 seeds. Qwen has a single run.

In [ ]:
def fool_rate_table(df, group_col):
    rows = []
    for g, part in df.groupby(group_col):
        row = {group_col: g, "pairs": len(part)}
        for name, label in [("deb_in", "DeBERTa (in-domain)"), ("deb_qqp", "DeBERTa (QQP)"),
                            ("deb_debias", "DeBERTa debiased (QQP)")]:
            rates = [(part[f"{name}_s{s}"] > 0.5).mean() for s in SEEDS]
            row[label] = f"{np.mean(rates):.3f} ± {np.std(rates, ddof=1):.3f}"
        row["Qwen (zero-shot)"] = f"{(part['qwen_prob'] > 0.5).mean():.3f}"
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
print("Fool rate by dataset (lower is better)")
by_dataset = fool_rate_table(clean.assign(all="all"), "dataset")
overall = fool_rate_table(clean.assign(all="all"), "all").rename(columns={"all": "dataset"})
by_dataset = pd.concat([by_dataset, overall], ignore_index=True)
by_dataset.to_csv(f"{OUT}/taskD_fool_rate_by_dataset.csv", index=False)
display(by_dataset)

In [ ]:
print("Fool rate by edit type")
by_edit = fool_rate_table(clean, "edit_type")
by_edit.to_csv(f"{OUT}/taskD_fool_rate_by_edit.csv", index=False)
display(by_edit)

### Standard vs debiased DeBERTa on the new trap pairs
Both models were trained on QQP and see exactly the same pairs, so we can use McNemar's test again, comparing each model's combined 5-seed prediction (average probability above 0.5). Here "right" means correctly saying "not a paraphrase".

In [ ]:
std_right = (clean["deb_qqp_prob"] <= 0.5).values
deb_right = (clean["deb_debias_prob"] <= 0.5).values

only_std = int(np.sum(std_right & ~deb_right))
only_deb = int(np.sum(~std_right & deb_right))
p = binomtest(only_std, only_std + only_deb, 0.5).pvalue if (only_std + only_deb) > 0 else 1.0

print(f"Correctly rejected: standard {std_right.mean():.3f} | debiased {deb_right.mean():.3f}")
print(f"Only standard right: {only_std} | only debiased right: {only_deb} | p-value: {p:.2g}")

## Accuracy, F1 and ROC on near-misses plus real paraphrases
On their own, the near-misses only have label 0, so F1 and ROC curves cannot be computed from them. Adding the control pairs fixes this: those are the dataset's real paraphrases of the same source sentences, with label 1. Together they make a balanced test set where both sides come from the same sentences. That makes it a hard test, because both kinds of pair share most of their words with the source, so a model that only looks at word overlap has nothing to go on.

Accuracy and F1 use the usual 0.5 cut-off and are averaged over the 5 seeds. AUC and the ROC curves use the probability averaged over the seeds.

In [ ]:
FIG_D = f"{OUT}/figures"
os.makedirs(FIG_D, exist_ok=True)

D_JUDGES = [("deb_in", "DeBERTa (in-domain)"), ("deb_qqp", "DeBERTa (QQP)"),
            ("deb_debias", "DeBERTa debiased (QQP)")]

# near-misses first (label 0), then the real paraphrases of the same sentences (label 1)
y_both = np.r_[np.zeros(len(clean)), np.ones(len(clean))]

both_probs = {}      # probability averaged over seeds, used for AUC and the ROC curves
rows = []
for name, label in D_JUDGES:
    # accuracy and F1 for each seed
    acc, f1 = [], []
    for s in SEEDS:
        p = np.r_[clean[f"{name}_s{s}"], clean[f"ref_{name}_s{s}"]]
        pred = (p > 0.5).astype(int)
        acc.append(accuracy_score(y_both, pred))
        f1.append(f1_score(y_both, pred))

    # averaged probability for AUC
    ref_mean = clean[[f"ref_{name}_s{s}" for s in SEEDS]].mean(axis=1)
    both_probs[label] = np.r_[clean[f"{name}_prob"], ref_mean]

    rows.append({"judge": label,
                 "accuracy": f"{np.mean(acc):.3f} ± {np.std(acc, ddof=1):.3f}",
                 "F1": f"{np.mean(f1):.3f} ± {np.std(f1, ddof=1):.3f}",
                 "AUC": round(roc_auc_score(y_both, both_probs[label]), 3)})


In [ ]:
# Qwen has a single run
both_probs["Qwen (zero-shot)"] = np.r_[clean["qwen_prob"], clean["ref_qwen_prob"]]
q_pred = (both_probs["Qwen (zero-shot)"] > 0.5).astype(int)
rows.append({"judge": "Qwen (zero-shot)",
             "accuracy": f"{accuracy_score(y_both, q_pred):.3f}",
             "F1": f"{f1_score(y_both, q_pred):.3f}",
             "AUC": round(roc_auc_score(y_both, both_probs["Qwen (zero-shot)"]), 3)})

d_metrics = pd.DataFrame(rows)
d_metrics.to_csv(f"{OUT}/taskD_accuracy_f1_auc.csv", index=False)
print("Near-misses + real paraphrases:", len(y_both), "pairs, half of each")
d_metrics

### ROC curves and the two rates side by side
The left plot shows the ROC curves: the higher a curve rises above the diagonal, the better that judge separates real paraphrases from near-misses at any cut-off. The right plot puts each judge's two rates next to each other. A good judge has a tall green bar (accepts real paraphrases) and a short red bar (rarely fooled).

In [ ]:
colors = {"DeBERTa (in-domain)": "green", "DeBERTa (QQP)": "blue",
          "DeBERTa debiased (QQP)": "orange", "Qwen (zero-shot)": "purple"}
labels = list(both_probs.keys())
n_pairs = len(clean)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# left: one ROC curve per judge
for label in labels:
    fpr, tpr, _ = roc_curve(y_both, both_probs[label])
    axes[0].plot(fpr, tpr, color=colors[label], lw=2,
                 label=f"{label} (AUC {roc_auc_score(y_both, both_probs[label]):.3f})")
axes[0].plot([0, 1], [0, 1], color="gray", linestyle="--", lw=1, label="coin flip")
axes[0].set_xlabel("False positive rate (near-misses called paraphrases)")
axes[0].set_ylabel("True positive rate (real paraphrases accepted)")
axes[0].set_title("ROC: near-misses vs real paraphrases")
axes[0].legend(fontsize=8)

# right: acceptance of real paraphrases next to the fool rate on near-misses
accept = [(both_probs[l][n_pairs:] > 0.5).mean() for l in labels]
fool = [(both_probs[l][:n_pairs] > 0.5).mean() for l in labels]
x = np.arange(len(labels))
axes[1].bar(x - 0.2, accept, 0.4, color="green", label="accepts real paraphrases (higher is better)")
axes[1].bar(x + 0.2, fool, 0.4, color="red", label="fooled by near-misses (lower is better)")
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, rotation=15, fontsize=8)
axes[1].set_ylim(0, 1)
axes[1].set_title("Real paraphrases vs near-misses")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(f"{FIG_D}/taskD_roc_and_rates.png", dpi=200)
plt.show()

### Confusion matrices
Each matrix shows, for one judge, what share of the near-misses and of the real paraphrases ended up in each predicted class. The top-right cell is the fool rate, and the bottom-left cell is the share of real paraphrases that were wrongly rejected.

In [ ]:
fig, axes = plt.subplots(1, len(labels), figsize=(4 * len(labels), 3.6))
for ax, label in zip(axes, labels):
    pred = (both_probs[label] > 0.5).astype(int)
    cm = confusion_matrix(y_both, pred, labels=[0, 1], normalize="true") * 100   # % of each true class
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=100)

    # write the percentage in each square
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]:.0f}%", ha="center", va="center",
                    color="white" if cm[i, j] > 50 else "black")

    ax.set_xticks([0, 1])
    ax.set_xticklabels(["not para", "para"])
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["near-miss", "real para"])
    ax.set_xlabel("Predicted")
    ax.set_title(label, fontsize=9)

plt.suptitle("Task D: % of each true class (averaged probability, cut-off 0.5)")
plt.tight_layout()
plt.savefig(f"{FIG_D}/taskD_confusion.png", dpi=200)
plt.show()

### Fool rate by edit type
This chart shows which kind of edit is hardest to catch. If swaps fool the models far more than number changes or negations, that points back to the word-overlap shortcut, since a swap keeps exactly the same words.

In [ ]:
edit_types = sorted(clean["edit_type"].unique())
judges = D_JUDGES + [("qwen", "Qwen (zero-shot)")]
width = 0.2

fig, ax = plt.subplots(figsize=(8, 4))
for j, (name, label) in enumerate(judges):
    rates = []
    for e in edit_types:
        part = clean[clean["edit_type"] == e]
        prob = part["qwen_prob"] if name == "qwen" else part[f"{name}_prob"]
        rates.append((prob > 0.5).mean())
    ax.bar(np.arange(len(edit_types)) + (j - 1.5) * width, rates, width, color=colors[label], label=label)

ax.set_xticks(np.arange(len(edit_types)))
ax.set_xticklabels(edit_types)
ax.set_ylim(0, 1)
ax.set_ylabel("Fool rate (lower is better)")
ax.set_title("Which kind of edit fools each judge most?")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{FIG_D}/taskD_fool_rate_by_edit.png", dpi=200)
plt.show()

## Saved files
- `taskD_generated.csv`: raw Qwen outputs
- `taskD_all_with_flags.csv`: every output with the filter checks
- `taskD_clean.csv`: the usable near-miss pairs (label 0)
- `taskD_handcheck.csv`: the 60 pairs for the hand check
- `taskD_text_metrics.csv`: overlap, BLEU and BERTScore by dataset and edit type
- `taskD_text_metrics_vs_real.csv`: BLEU and BERTScore for near-misses vs the real human paraphrases, by dataset
- `taskD_scores.csv`: every judge's probability for every pair (near-misses and real paraphrases), along with the BLEU and BERTScore values
- `taskD_control.csv`: each judge's acceptance rate on real paraphrases next to its fool rate on near-misses
- `taskD_fool_rate_by_dataset.csv`, `taskD_fool_rate_by_edit.csv`: the main results
- `taskD_metadata.json`: model version, prompts and settings
- `taskD_accuracy_f1_auc.csv`: accuracy, F1 and AUC on near-misses plus real paraphrases
- `figures/`: ROC curves, the two rates side by side, confusion matrices and fool rate by edit type

All GPU results are stored in `taskD_scores.csv`, so the tables above can be rebuilt on CPU without loading any model.

The `taskD_clean.csv` file can also go to the Phi-4 judge, which is run separately.

# Task C: New sentences

Every sentence used so far comes from MRPC, QQP or PAWS, which are the same datasets the DeBERTa models were trained on. Task C checks whether the classifiers still recognise paraphrases when the text comes from somewhere else.

In this part, Qwen writes around 300 new sentences that are not taken from any of the three datasets. It then paraphrases each one with the same prompt used in Task A. Every pair is therefore a paraphrase (label 1), so the main number is the **acceptance rate**: how often each judge correctly calls the pair a paraphrase. A higher acceptance rate is better.

Two caveats apply. First, Qwen writes the sentence, paraphrases it and then judges the pair, so its own score here is the most likely to be biased in its favour. Second, "new" means new to the three datasets, not new to Qwen, since the sentences come from Qwen itself. What this task really tests is whether the trained classifiers generalise beyond the benchmarks they were trained on.

This part reuses the model, the settings and the helper functions defined in Task D above.

In [ ]:
OUT_C = f"/home/{RUN_USER}/NLP_Project/qwen_generation/taskC"
os.makedirs(OUT_C, exist_ok=True)

## Writing the new source sentences
To avoid favouring one dataset, the sentences are written in 3 styles, each matching one of the datasets:
- **question:** a question someone might post online (like QQP)
- **news:** a sentence from a news report (like MRPC)
- **encyclopedia:** a factual sentence from an encyclopedia article (like PAWS)

Each style is combined with 10 everyday topics. Qwen writes 30 sentences for every style and topic pair, since many come out as repeats and are removed in the next steps. After cleaning, 10 are kept per pair, so each style has up to 100 sentences.

The facts in these sentences don't need to be true, since only the paraphrase relation between the two sentences matters.

Qwen tends to repeat itself at its normal temperature, so the source sentences (and only those) are written at temperature 1.0, and 30 are drawn per style and topic so that enough different ones are left after removing repeats.

In [ ]:
STYLES = {
    "question": "Write one new question that someone might ask on a Q&A website like Quora, about the given topic.",
    "news": "Write one new sentence in the style of a news report about the given topic. Include a specific name, place, date or number.",
    "encyclopedia": "Write one new factual sentence in the style of an encyclopedia about the given topic. Include a specific name, place, date or number.",
}
SRC_COMMON = " Make it original and 8 to 30 words long. Write in English and output only the sentence."

TOPICS = ["sport", "science", "history", "food and cooking", "technology",
          "health", "travel", "music", "business", "nature and animals"]
PER_CELL = 30        # sentences per style and topic
KEEP_PER_CELL = 10
SRC_TEMP = 1.0       # higher temperature only for writing the new sources, so they repeat less

In [ ]:
SRC_FILE = f"{OUT_C}/taskC_sources_raw.csv"

if os.path.exists(SRC_FILE):
    c_raw = pd.read_csv(SRC_FILE, keep_default_na=False)
    print("Loaded saved source sentences:", len(c_raw))
else:
    rows = []
    for k, (style, instruction) in enumerate(STYLES.items()):
        # the same request 30 times per topic, sampling makes every answer different
        inputs = [f"Topic: {t}" for t in TOPICS for _ in range(PER_CELL)]
        topics = [t for t in TOPICS for _ in range(PER_CELL)]

        torch.manual_seed(SEED + 1000 + k)          # fixed seed for each style
        outs = generate_batch(instruction + SRC_COMMON, inputs, temperature=SRC_TEMP)

        for t, o in zip(topics, outs):
            rows.append({"style": style, "topic": t, "source": o})
        print(f"{style}: {len(outs)} sentences written")

    c_raw = pd.DataFrame(rows)
    c_raw.to_csv(SRC_FILE, index=False)

c_raw.head(6)

## Checking the new sentences
Before paraphrasing, each sentence is checked, and it is removed if:
- it is empty or starts with extra text such as "Here is…"
- it doesn't match its style (a question that doesn't end with "?", or a news or encyclopedia sentence that does)
- it is too short or too long (fewer than 6 or more than 40 words)
- it is almost the same as another generated sentence (Jaccard overlap of 0.6 or more)
- it already appears in MRPC, QQP or PAWS, either word for word or as a very close match (TF-IDF cosine similarity of 0.8 or more with any sentence in any split)

The last check is what makes these sentences genuinely new to the three datasets.

In [ ]:
c_raw["source"] = c_raw["source"].astype(str).str.strip()
src_s = c_raw["source"]

checks = pd.DataFrame(index=c_raw.index)
checks["empty_or_extra"] = (src_s == "") | src_s.str.contains(r"^(?:here|sure|certainly)\b|sentence:",
                                                              case=False, regex=True)

# questions should end with "?" and the other two styles should not
is_question = src_s.str.endswith("?")
checks["wrong_style"] = ((c_raw["style"] == "question") & ~is_question) | ((c_raw["style"] != "question") & is_question)

n_words = src_s.apply(lambda x: len(words(x)))
checks["bad_length"] = (n_words < 6) | (n_words > 40)

# For near-duplicates among the generated sentences: keep the first one, drop later ones that are too similar
dup = []
kept_so_far = []
for text in src_s:
    is_dup = any(jaccard(text, other) >= 0.6 for other in kept_so_far)
    dup.append(is_dup)
    if not is_dup:
        kept_so_far.append(text)
checks["near_duplicate"] = dup

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# every sentence from every split of the three datasets
all_sents = []
for ds in ["mrpc", "qqp", "paws"]:
    for sp in ["train", "dev", "test"]:
        df = pd.read_csv(f"{SPLITS}/{ds}_{sp}.csv", keep_default_na=False)
        all_sents += df["sentence1"].astype(str).tolist() + df["sentence2"].astype(str).tolist()
print("Dataset sentences to compare against:", len(all_sents))

In [ ]:
# 1) exact matches, ignoring case and punctuation
known = set(" ".join(words(x)) for x in all_sents)
checks["in_dataset"] = [" ".join(words(x)) in known for x in src_s]

# 2) very close matches: highest TF-IDF cosine similarity with any dataset sentence
vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit(all_sents + src_s.tolist())
X_data = vec.transform(all_sents)
X_new = vec.transform(src_s)

In [ ]:
max_sim = []
for i in range(0, X_new.shape[0], 50):           # 50 sentences at a time to save memory
    sims = X_new[i:i + 50] @ X_data.T
    max_sim += sims.max(axis=1).toarray().ravel().tolist()
c_raw["max_sim_to_data"] = np.round(max_sim, 3)
checks["close_to_dataset"] = c_raw["max_sim_to_data"] >= 0.8

In [ ]:
c_raw["ok"] = ~checks.any(axis=1)

# how many sentences each check removed, per style
src_summary = checks.groupby(c_raw["style"]).sum()
src_summary["ok"] = c_raw.groupby("style")["ok"].sum()
display(src_summary)

# keep 10 per style and topic
c_src = c_raw[c_raw["ok"]].groupby(["style", "topic"]).head(KEEP_PER_CELL).reset_index(drop=True)
c_src.to_csv(f"{OUT_C}/taskC_sources.csv", index=False)
print("Source sentences kept:", len(c_src))
print(c_src.groupby("style").size())

To show that the sentences are varied and not near-copies of each other, the table below gives the average length and the distinct-2 score for each style. Distinct-2 is the share of word pairs (bigrams) that are unique across all sentences of that style, so a value closer to 1 means more variety.

In [ ]:
def distinct2(texts):
    # unique word pairs / all word pairs
    bigrams = []
    for t in texts:
        w = words(t)
        bigrams += list(zip(w, w[1:]))
    return len(set(bigrams)) / max(len(bigrams), 1)

In [ ]:
variety = []
for style, part in c_src.groupby("style"):
    variety.append({"style": style, "sentences": len(part),
                    "avg words": round(part["source"].apply(lambda x: len(words(x))).mean(), 1),
                    "distinct-2": round(distinct2(part["source"]), 3),
                    "max similarity to datasets": part["max_sim_to_data"].max()})
pd.DataFrame(variety)

## Paraphrasing the new sentences
Each new sentence is paraphrased with a prompt based on the team's Task A prompt and the same generation settings, so the results can be compared with Task A. The prompt was shortened, and the rule about questions now works both ways, because Qwen was turning statements into questions.

In [ ]:
# paraphrase prompt based on the team's Task A prompt
PARA_PROMPT = ("Rewrite the sentence with different words and structure, but keep the exact same meaning. "
               "Keep all names, numbers, places, dates and relations unchanged, and do not add or remove information. "
               "A statement must stay a statement, and a question must stay a question; do not answer it. "
               "Write in English and output only the paraphrase.")

PARA_FILE = f"{OUT_C}/taskC_paraphrased.csv"

if os.path.exists(PARA_FILE):
    c_pairs = pd.read_csv(PARA_FILE, keep_default_na=False)
    print("Loaded saved paraphrases:", len(c_pairs))
else:
    outs = []
    for i in range(0, len(c_src), BS):
        torch.manual_seed(SEED + 2000 + i // BS)     # fixed seed for each batch
        outs += generate_batch(PARA_PROMPT, c_src["source"].iloc[i:i + BS].tolist())
    c_pairs = c_src.assign(paraphrase=outs)
    c_pairs.to_csv(PARA_FILE, index=False)
    print("Paraphrased", len(c_pairs), "sentences")

meta_c = {"task": "C: new sentences and their paraphrases", "model": HF_ID, "revision": REVISION,
          "dtype": "bfloat16", "gen_config": GEN_CFG, "seed": SEED, "styles": STYLES,
          "source_prompt_suffix": SRC_COMMON, "topics": TOPICS, "paraphrase_prompt": PARA_PROMPT,
          "date": datetime.datetime.now().isoformat(timespec="seconds")}
json.dump(meta_c, open(f"{OUT_C}/taskC_metadata.json", "w"), indent=1)

## Cleaning the paraphrases
The same kind of checks as before are applied. A pair is removed if the paraphrase is empty, an exact copy of the source, starts with extra text, changes the sentence type (a question becomes a statement, or the other way round), or has a very different length (less than 0.5 or more than 2 times the source). The length range is wider than in Task D, because a good paraphrase can legitimately be a bit shorter or longer than the original.

In [ ]:
# checking if every paraphrase is text, with no extra spaces at the ends
c_pairs["paraphrase"] = c_pairs["paraphrase"].astype(str).str.strip()
a = c_pairs["source"]
b = c_pairs["paraphrase"]

# one True/False column per check, where True means the pair fails that check
pflags = pd.DataFrame(index=c_pairs.index)

# no output was returned
pflags["empty"] = b == ""

# the output is identical to the source, so no rewording took place
pflags["copy"] = b.str.lower() == a.str.lower()

# the output includes extra text
pflags["extra_text"] = b.str.contains(r"^(?:here|sure|certainly)\b|paraphrase:", case=False, regex=True)

# the sentence type changed (a question became a statement, or the other way round)
pflags["type_changed"] = a.str.endswith("?") != b.str.endswith("?")

# the length changed too much (less than half or more than double the source)
ratio = b.str.len() / a.str.len()
pflags["length_change"] = (ratio < 0.5) | (ratio > 2.0)

# a pair is kept only if it passes every check
c_pairs["keep"] = ~pflags.any(axis=1)

# number of pairs removed by each check and the number kept, for each style
display(pflags.groupby(c_pairs["style"]).sum().assign(kept=c_pairs.groupby("style")["keep"].sum()))

In [ ]:
c_clean = c_pairs[c_pairs["keep"]].reset_index(drop=True)
c_clean["label"] = 1      # every pair is a paraphrase by design
c_clean["overlap"] = [jaccard(x, y) for x, y in zip(c_clean["source"], c_clean["paraphrase"])]
c_clean.to_csv(f"{OUT_C}/taskC_clean.csv", index=False)
print("Usable paraphrase pairs:", len(c_clean))

In [ ]:
# a few examples to eyeball
for _, r in c_clean.sample(6, random_state=SEED).iterrows():
    print(f"\n[{r['style']} | {r['topic']}]\n  SRC: {r['source']}\n  GEN: {r['paraphrase']}")

## Hand check
The automatic checks only catch obvious problems, so a small sample is also checked by hand, just like in Task D. This time it's 17 pairs per style, 51 in total. They are saved to `taskC_handcheck.csv` with an empty `same_meaning` column. Each pair gets a 1 if the paraphrase keeps the meaning and a 0 if it doesn't. After filling it in, running the next cell again gives the result. The file is only created once, so the answers won't be overwritten.

In [ ]:
HAND_C = f"{OUT_C}/taskC_handcheck.csv"

if not os.path.exists(HAND_C):
    sample = c_clean.groupby("style").sample(n=17, random_state=SEED)
    sample = sample[["style", "topic", "source", "paraphrase"]].copy()
    sample["same_meaning"] = ""
    sample.to_csv(HAND_C, index=False)
    print("Saved", len(sample), "pairs to", HAND_C, "for checking")

hand_c = pd.read_csv(HAND_C, keep_default_na=False)
filled = hand_c[hand_c["same_meaning"].astype(str).str.strip() != ""]
if len(filled) > 0:
    filled = filled.assign(same_meaning=filled["same_meaning"].astype(int))
    print(f"Checked {len(filled)} pairs: meaning kept in {filled['same_meaning'].mean():.1%}")
    display(filled.groupby("style")["same_meaning"].mean().round(2))
else:
    print("Hand check not filled in yet")

## Text metrics
The same two metrics as in Task D are used, but here the hoped-for result is the opposite:
- **BERTScore F1:** should be high, because a paraphrase should keep the meaning.
- **BLEU:** should be low, because a good paraphrase rewords the sentence rather than copying it.

High BERTScore together with low BLEU is the sign of a real paraphrase: same meaning, new wording.

In [ ]:
c_clean["bleu"] = [sacrebleu.sentence_bleu(y, [x]).score for x, y in zip(c_clean["source"], c_clean["paraphrase"])]

_, _, F1 = bert_score(c_clean["paraphrase"].tolist(), c_clean["source"].tolist(),
                      lang="en", batch_size=64, device="cuda", verbose=False)
c_clean["bertscore_f1"] = F1.numpy()

c_metrics = c_clean.groupby("style")[["overlap", "bleu", "bertscore_f1"]].mean()
c_metrics.loc["all"] = c_clean[["overlap", "bleu", "bertscore_f1"]].mean()
c_metrics = c_metrics.round(3)
c_metrics.to_csv(f"{OUT_C}/taskC_text_metrics.csv")
c_metrics

## Do the classifiers accept the new paraphrases?
Since these sentences don't come from any of the training sets, there isn't an "in-domain" DeBERTa for them. So every pair is scored by all three standard DeBERTa models (trained on MRPC, QQP and PAWS), plus the debiased DeBERTa and Qwen. Having all three DeBERTa models also makes it possible to see whether each one does best on the style closest to what it was trained on, for example the QQP model on the question-style sentences.

One small detail: the MRPC and PAWS models were trained on Treebank-style text, where punctuation is split off. To keep things fair, both sentences are converted to that style before these two models score them. The QQP-trained models and Qwen get the normal text, since that's what QQP looks like.

In [ ]:
def styled(texts, trained_on):
    # MRPC and PAWS models were trained on Treebank-style text, so they get the same style here
    if trained_on in ("mrpc", "paws"):
        return [normalise(t, trained_on) for t in texts]
    return list(texts)

ca = c_clean["source"].tolist()
cb = c_clean["paraphrase"].tolist()

for s in SEEDS:
    for ds in ["mrpc", "qqp", "paws"]:
        c_clean[f"deb_{ds}_s{s}"] = deberta_probs(checkpoints[(ds, s)], styled(ca, ds), styled(cb, ds))
    c_clean[f"deb_debias_s{s}"] = deberta_probs(checkpoints[("qqp_debias", s)], ca, cb)
    print("seed", s, "done")

c_clean["qwen_prob"] = qwen_probs(ca, cb)
c_clean.to_csv(f"{OUT_C}/taskC_scores.csv", index=False)
print("Scores saved for", len(c_clean), "pairs")

## Results: acceptance rate
The acceptance rate is the share of pairs a judge calls a paraphrase (probability above 0.5). For the DeBERTa models it is shown as mean ± std over the 5 seeds. Qwen has a single run.

In [ ]:
C_JUDGES = [("deb_mrpc", "DeBERTa (MRPC)"), ("deb_qqp", "DeBERTa (QQP)"),
            ("deb_paws", "DeBERTa (PAWS)"), ("deb_debias", "DeBERTa debiased (QQP)")]

def accept_table(df, group_col):
    rows = []
    for g, part in df.groupby(group_col):
        row = {group_col: g, "pairs": len(part)}
        for name, label in C_JUDGES:
            rates = [(part[f"{name}_s{s}"] > 0.5).mean() for s in SEEDS]
            row[label] = f"{np.mean(rates):.3f} ± {np.std(rates, ddof=1):.3f}"
        row["Qwen (zero-shot)"] = f"{(part['qwen_prob'] > 0.5).mean():.3f}"
        rows.append(row)
    return pd.DataFrame(rows)

c_results = pd.concat([accept_table(c_clean, "style"),
                       accept_table(c_clean.assign(style="all"), "style")], ignore_index=True)
c_results.to_csv(f"{OUT_C}/taskC_acceptance.csv", index=False)
print("Acceptance rate on new paraphrases (higher is better)")
c_results

## Charts
The first chart shows how often each judge accepts the new paraphrases, split by style. If a DeBERTa model does clearly better on the style closest to its own training data (for example the QQP model on questions), that suggests it depends on the look of its training text and not only on meaning.

In [ ]:
FIG_C = f"{OUT_C}/figures"
os.makedirs(FIG_C, exist_ok=True)

c_colors = {"DeBERTa (MRPC)": "green", "DeBERTa (QQP)": "blue", "DeBERTa (PAWS)": "brown",
            "DeBERTa debiased (QQP)": "orange", "Qwen (zero-shot)": "purple"}
styles = sorted(c_clean["style"].unique())
c_judges = C_JUDGES + [("qwen", "Qwen (zero-shot)")]
width = 0.16

fig, ax = plt.subplots(figsize=(9, 4))
for j, (name, label) in enumerate(c_judges):
    rates = []
    for st in styles:
        part = c_clean[c_clean["style"] == st]
        if name == "qwen":
            rates.append((part["qwen_prob"] > 0.5).mean())
        else:
            # average of the 5 seeds
            rates.append(np.mean([(part[f"{name}_s{s}"] > 0.5).mean() for s in SEEDS]))
    ax.bar(np.arange(len(styles)) + (j - 2) * width, rates, width, color=c_colors[label], label=label)

ax.set_xticks(np.arange(len(styles)))
ax.set_xticklabels(styles)
ax.set_ylim(0, 1)
ax.set_ylabel("Acceptance rate (higher is better)")
ax.set_title("Task C: new paraphrases accepted by each judge, by style")
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig(f"{FIG_C}/taskC_acceptance_by_style.png", dpi=200)
plt.show()

### Paraphrases vs near-misses on the text metrics
This plot brings Tasks C and D together. Every pair is placed by its BLEU score (how many words were copied from the source) and its BERTScore (how close the meaning is). Real paraphrases, whether written by Qwen or by people, are expected to sit on the left with low BLEU, while the near-misses are expected to sit on the right with high BLEU. Wherever the groups overlap, the text metrics alone can't tell a paraphrase from a trap.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))

# Qwen's paraphrases of the new sentences (Task C)
ax.scatter(c_clean["bleu"], c_clean["bertscore_f1"], s=12, alpha=0.5, color="green",
           label=f"Task C: Qwen paraphrases of new sentences (n={len(c_clean)})")

# the dataset's own human paraphrases (Task D control)
ax.scatter(clean["ref_bleu"], clean["ref_bertscore_f1"], s=12, alpha=0.5, color="blue",
           label=f"Task D: real human paraphrases (n={len(clean)})")

# the near-misses (Task D)
ax.scatter(clean["bleu"], clean["bertscore_f1"], s=12, alpha=0.5, color="red",
           label=f"Task D: near-misses (n={len(clean)})")

ax.set_xlabel("BLEU vs source (words copied)")
ax.set_ylabel("BERTScore F1 vs source (meaning)")
ax.set_title("Paraphrases vs near-misses")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{FIG_C}/bleu_vs_bertscore.png", dpi=200)
plt.show()

## Saved files (Task C)
- `taskC_sources_raw.csv`: every sentence Qwen wrote, before checking
- `taskC_sources.csv`: the new source sentences that passed the checks
- `taskC_paraphrased.csv`: the sources with Qwen's paraphrases
- `taskC_clean.csv`: the usable paraphrase pairs (label 1)
- `taskC_handcheck.csv`: the 51 pairs for the hand check
- `taskC_text_metrics.csv`: BLEU and BERTScore by style
- `taskC_scores.csv`: every judge's probability for every pair
- `taskC_acceptance.csv`: the main results
- `taskC_metadata.json`: model version, prompts and settings
- `figures/`: acceptance rate by style, and BLEU vs BERTScore for Task C paraphrases, real paraphrases and near-misses

The `taskC_clean.csv` file can also go to the Phi-4 judge. Comparing these acceptance rates with the ones from Task A (same paraphrase prompt, but dataset sentences) shows whether the classifiers do worse on text from outside their training data.